In [1]:
import os, json, yaml

# configs 创建
configs/base_to_novel/xxx_base_to_novel.yaml


In [2]:
import os
import yaml

config_dir = '/root/wj/EZ_CLIP/configs/base_to_novel'
dataset = ['HARDVS', 'PAF', 'DVS128Gesture', 'SeAct']
config_file_templates = ['{}_base_to_novel.yaml']

for d in dataset:
    for config_template in config_file_templates:
        config_file = os.path.join(config_dir, config_template.format(d))
        print(config_file)
        # Read existing YAML content
        if os.path.exists(config_file):
            with open(config_file, 'r') as infile:
                config_data = yaml.safe_load(infile) or {}
        else:
            config_file_base = os.path.join(config_dir, 'basic_base_to_novel.yaml')
            with open(config_file_base, 'r') as infile:
                config_data = yaml.safe_load(infile) or {}
        # basic setting
        config_data['network']['type'] = 'base_to_novel' 
        config_data['weight_save_dir'] = '/root/autodl-tmp/SAMPLE'
        config_data['training_name'] = f"{config_template.format(d).replace('.yaml', '')}_base_to_novel"
        config_data['network']['sim_header'] = 'Transf'
        config_data['mm_prompt']['CTX_INIT'] = 'a human action of'
        config_data['pretrain'] = None
        config_data['resume'] = None
        config_data['T_Adapter'] = False
        config_data['data']['dataset'] = d
        config_data['solver']['epochs'] = 150
        # time prompt
        config_data['prompt']['use']= True
        config_data['prompt']['DEEP']= True
        # mm prompt
        config_data['mm_prompt']['use'] = True
        config_data['mm_prompt']['N_CTX']= 2
        config_data['mm_prompt']['PROMPT_DEPTH']= 9

        # Print log directory
        logdir = os.path.join(
            config_data["weight_save_dir"],
            config_data["network"]["type"],
            config_data["network"].get("arch", "unknown_arch"),
            config_data["data"].get("dataset", d),
            config_data['training_name']
        )
        # base_to_novel & create relative path file
        config_data['data']['train_list'] = f'dataset_splits/{d}/base_to_novel/{d}_base_train.txt'
        config_data['data']['val_list'] = f'dataset_splits/{d}/base_to_novel/{d}_base_val.txt'
        config_data['data']['gpt_discription'] = f'GPT_discription/{d}_gpt_Class_discription_new.csv'
        config_data['data']['label_list'] = f'lists/{d}_labels.csv'
        config_data['data']['novel_val_list'] = f'dataset_splits/{d}/base_to_novel/{d}_novel_val.txt'
        config_data['data']['novel_gpt_discription'] = f'GPT_discription/{d}_gpt_Class_discription_new.csv'
        config_data['data']['novel_label_list'] = f'lists/{d}_labels.csv'

        # 创建必要的文件和目录
        base_to_novel_dir = f'/root/wj/EZ_CLIP/dataset_splits/{d}/base_to_novel'
        os.makedirs(base_to_novel_dir, exist_ok=True)
        
        files_to_create = [
            f'{d}_base_train.txt',
            f'{d}_base_val.txt',
            f'{d}_novel_val.txt',
            f'{d}_novel_train.txt',
            f'{d}_novel_val_class_list.csv',
            f'{d}_base_train_class_list.csv'
        ]
        
        for file in files_to_create:
            file_path = os.path.join(base_to_novel_dir, file)
            if not os.path.exists(file_path):
                with open(file_path, 'w') as f:
                    pass  # 创建空文件
        
        # 创建GPT描述文件
        gpt_description_dir = '/root/wj/EZ_CLIP/GPT_discription'
        os.makedirs(gpt_description_dir, exist_ok=True)
        gpt_description_file = f'{d}_gpt_Class_discription_new.csv'
        gpt_description_path = os.path.join(gpt_description_dir, gpt_description_file)
        if not os.path.exists(gpt_description_path):
            with open(gpt_description_path, 'w') as f:
                pass  # 创建空文件
        
        print(f"已为数据集 {d} 创建所需的文件和目录")
        
        # Write back to YAML file
        with open(config_file, 'w') as outfile:
            yaml.dump(config_data, outfile, default_flow_style=False)

/root/wj/EZ_CLIP/configs/base_to_novel/HARDVS_base_to_novel.yaml
已为数据集 HARDVS 创建所需的文件和目录
/root/wj/EZ_CLIP/configs/base_to_novel/PAF_base_to_novel.yaml
已为数据集 PAF 创建所需的文件和目录
/root/wj/EZ_CLIP/configs/base_to_novel/DVS128Gesture_base_to_novel.yaml
已为数据集 DVS128Gesture 创建所需的文件和目录
/root/wj/EZ_CLIP/configs/base_to_novel/SeAct_base_to_novel.yaml
已为数据集 SeAct 创建所需的文件和目录


# Create the following files
dataset_split/xxx/base_to_novel/xxx_base_train.txt

dataset_split/xxx/base_to_novel/xxx_base_val.txt 

dataset_split/xxx/base_to_novel/xxx_novel_val.txt

dataset_split/xxx/base_to_novel/xxx_novel_train.txt

dataset_split/xxx/base_to_novel/xxx_novel_val_class_list.csv

dataset_split/xxx/base_to_novel/xxx_base_train_class_list.csv


In [9]:
## 1. 划分base类和novel类
## 方法：根据频率划分，频率高的一半是base类，频率低的一半是novel类，主要依据以下文件
## (1) lists/xxx_labels.csv
### 格式：
### id,name
### 0,L type clamp back
## (2) dataset_splits/xxx/zero-shot/train.txt
## (3) dataset_splits/xxx/zero-shot/val.txt
## format:
### path, jpg_files_number, label
### /root/autodl-tmp/DVS128Gesture_Sampled_EZCLIP/hand_clapping/user01_fluorescent 16 0
import csv
from collections import defaultdict

def split_base_novel(base_dir, label_file, train_file, val_file, base_train_file, base_val_file, novel_train_file, novel_val_file, base_class_list_file, novel_class_list_file):
    # abspath
    label_file = os.path.join(base_dir, label_file)
    train_file = os.path.join(base_dir, train_file)
    val_file = os.path.join(base_dir, val_file)
    base_train_file = os.path.join(base_dir, base_train_file)
    base_val_file = os.path.join(base_dir, base_val_file)
    novel_train_file = os.path.join(base_dir, novel_train_file)
    novel_val_file = os.path.join(base_dir, novel_val_file)
    base_class_list_file = os.path.join(base_dir, base_class_list_file)
    novel_class_list_file = os.path.join(base_dir, novel_class_list_file)
    # 读取标签文件
    with open(label_file, 'r') as f:
        reader = csv.reader(f)
        # escape the first line
        next(reader)
        labels = {int(row[0]): row[1] for row in reader}

    # 统计每个类别的频率
    freq = defaultdict(int)
    for file in [train_file, val_file]:
        with open(file, 'r') as f:
            for line in f:
                label = int(line.strip().split()[-1])
                freq[label] += 1

    # 按频率排序并划分base类和novel类
    sorted_labels = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    mid = len(sorted_labels) // 2
    base_classes = {label for label, _ in sorted_labels[:mid]}
    novel_classes = {label for label, _ in sorted_labels[mid:]}

    # 写入base类和novel类的类列表文件
    with open(base_class_list_file, 'w') as f:
        writer = csv.writer(f)
        for label in base_classes:
            writer.writerow([label, labels[label]])

    with open(novel_class_list_file, 'w') as f:
        writer = csv.writer(f)
        for label in novel_classes:
            writer.writerow([label, labels[label]])

    # 根据类别划分训练和验证集
    def write_split_file(input_file, base_output_file, novel_output_file):
        with open(input_file, 'r') as f, open(base_output_file, 'w') as base_f, open(novel_output_file, 'w') as novel_f:
            for line in f:
                label = int(line.strip().split()[-1])
                if label in base_classes:
                    base_f.write(line)
                else:
                    novel_f.write(line)

    write_split_file(train_file, base_train_file, novel_train_file)
    write_split_file(val_file, base_val_file, novel_val_file)

# 示例调用
for d in dataset:
    split_base_novel(
    base_dir = '/root/wj/EZ_CLIP',
    label_file='lists/{}_labels.csv'.format(d),
    train_file='dataset_splits/{}/Zero-shot/train.txt'.format(d),
    val_file='dataset_splits/{}/Zero-shot/val.txt'.format(d),
    base_train_file='dataset_splits/{}/base_to_novel/{}_base_train.txt'.format(d,d),
    base_val_file='dataset_splits/{}/base_to_novel/{}_base_val.txt'.format(d,d),
    novel_train_file='dataset_splits/{}/base_to_novel/{}_novel_train.txt'.format(d,d),
    novel_val_file='dataset_splits/{}/base_to_novel/{}_novel_val.txt'.format(d,d),
    base_class_list_file='dataset_splits/{}/base_to_novel/{}_base_train_class_list.csv'.format(d,d),
    novel_class_list_file='dataset_splits/{}/base_to_novel/{}_novel_val_class_list.csv'.format(d,d)
)
